# Experimental test 1 Result

* vllm기준으로 accuracy와 f1 score를 테스트
* few-shot테스트를 위해서 sc.Self_Consistency의 두번째 파라미터를 1, 2, 3, 4 로 변경하여 실험 수행
* 이외의 파라미터는 고정 
    * fewshot 테스트에 활용할 질문의 개수  : 60
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 5회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [80]:
import sys, os
import re
import numpy as np
from sklearn import metrics
import pandas as pd



In [94]:
# /mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_1/sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv
def sc_calc_acc_condition_with_temp_with_sc(run_id, llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    acc_list = []
    path = f'/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/{run_id}'
    file_list = os.listdir(path)
    print(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['result'] = tmp['result_long'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)
            tmp['answer'] = tmp['answer'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)

            
            eval_df = tmp.groupby(['id', 'answer', 'result']).count()['question'].reset_index().rename(columns={'question' : 'count'})
            eval_df = eval_df.sort_values(by = ['id', 'count'], ascending=[True, False]).groupby(['id', 'answer']).head(1)
            eval_df['equal_yn'] = np.where(eval_df['answer']==eval_df['result'], 1, 0)
            acc = (eval_df['equal_yn'].sum()/eval_df.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, eval_df], axis =0)
            
        df['equal_yn'] = np.where(df['answer']==df['result'], 1, 0)
        y_true = df['result']
        y_pred = df['answer']
        print(f"value count for golen : {df['answer'].value_counts()}")
        print(f"value count for o_result : {df['result'].value_counts()}")
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list, df


In [ ]:
# run_id_6/sc_vq_result_3_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv

list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('run_id_6', 'vq', 3, 30, 'Y', 30, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

sc_vq_result_3_30_Y_30_sys_prompt10_5_0.01_ver7
value count for golen : answer
1    157
0     96
2     47
Name: count, dtype: int64
value count for o_result : result
1    140
0     94
2     66
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.906     0.926     0.916        94
           1      0.790     0.886     0.835       140
           2      0.851     0.606     0.708        66

    accuracy                          0.837       300
   macro avg      0.849     0.806     0.820       300
weighted avg      0.840     0.837     0.832       300

vq_result_3_30_Y :  83.66666666666667
[np.float64(80.0), np.float64(66.66666666666666), np.float64(80.0), np.float64(100.0), np.float64(86.66666666666667), np.float64(83.33333333333334), np.float64(86.66666666666667), np.float64(76.66666666666667), np.float64(83.33333333333334), np.float64(93.33333333333333)]


In [103]:
# run_id_6/sc_vq_result_3_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv

list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('run_id_6', 'vq', 3, 30, 'Y', 30, 'sys_prompt14', 5,  0.01, 'ver7')
print(list_)

sc_vq_result_3_30_Y_30_sys_prompt14_5_0.01_ver7
value count for golen : answer
1    148
0    107
2     45
Name: count, dtype: int64
value count for o_result : result
1    122
0    112
2     66
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.953     0.911     0.932       112
           1      0.770     0.934     0.844       122
           2      0.933     0.636     0.757        66

    accuracy                          0.860       300
   macro avg      0.886     0.827     0.844       300
weighted avg      0.874     0.860     0.858       300

vq_result_3_30_Y :  86.0
[np.float64(90.0), np.float64(86.66666666666667), np.float64(73.33333333333333), np.float64(90.0), np.float64(80.0), np.float64(90.0), np.float64(83.33333333333334), np.float64(90.0), np.float64(86.66666666666667), np.float64(90.0)]


In [105]:
# run_id_6/sc_vq_result_3_30_Y_30_sys_prompt16_5_0.01_ver7_0.csv

list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('run_id_6', 'vq', 3, 30, 'Y', 30, 'sys_prompt16', 5,  0.01, 'ver7')
print(list_)

sc_vq_result_3_30_Y_30_sys_prompt16_5_0.01_ver7
value count for golen : answer
1    53
0    23
2    14
Name: count, dtype: int64
value count for o_result : result
1    42
2    27
0    21
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.826     0.905     0.864        21
           1      0.717     0.905     0.800        42
           2      1.000     0.519     0.683        27

    accuracy                          0.789        90
   macro avg      0.848     0.776     0.782        90
weighted avg      0.827     0.789     0.780        90

vq_result_3_30_Y :  78.88888888888889
[np.float64(70.0), np.float64(83.33333333333334), np.float64(83.33333333333334)]


In [100]:
df_[df_['id'] == 79076968]

,id,answer,result,count,equal_yn
36,79076968,1,2,4,0


In [113]:
# run_id_6/sc_vq_result_3_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv

list_, df_=         sc_calc_acc_condition_with_temp_with_sc('run_id_6', 'vq', 3, 30, 'Y', 30, 'sys_prompt16', 5,  0.01, 'ver7')
print(list_)

sc_vq_result_3_30_Y_30_sys_prompt16_5_0.01_ver7
value count for golen : answer
1    160
0     97
2     43
Name: count, dtype: int64
value count for o_result : result
1    134
0    103
2     63
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.959     0.903     0.930       103
           1      0.806     0.963     0.878       134
           2      0.977     0.667     0.792        63

    accuracy                          0.880       300
   macro avg      0.914     0.844     0.867       300
weighted avg      0.894     0.880     0.878       300

vq_result_3_30_Y :  88.0
[np.float64(90.0), np.float64(86.66666666666667), np.float64(90.0), np.float64(83.33333333333334), np.float64(90.0), np.float64(83.33333333333334), np.float64(86.66666666666667), np.float64(86.66666666666667), np.float64(93.33333333333333), np.float64(90.0)]


In [127]:
# run_id_6/sc_vq_result_3_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv

list_, df_=         sc_calc_acc_condition_with_temp_with_sc('run_id_6', 'vq', 3, 30, 'Y', 30, 'sys_prompt16', 5,  0.01, 'ver8')
print(list_)

sc_vq_result_3_30_Y_30_sys_prompt16_5_0.01_ver8
value count for golen : answer
1    140
2     84
0     76
Name: count, dtype: int64
value count for o_result : result
1    114
2    106
0     80
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.961     0.912     0.936        80
           1      0.757     0.930     0.835       114
           2      0.940     0.745     0.832       106

    accuracy                          0.860       300
   macro avg      0.886     0.863     0.867       300
weighted avg      0.876     0.860     0.861       300

vq_result_3_30_Y :  86.0
[np.float64(80.0), np.float64(90.0), np.float64(90.0), np.float64(90.0), np.float64(83.33333333333334), np.float64(83.33333333333334), np.float64(83.33333333333334), np.float64(83.33333333333334), np.float64(93.33333333333333), np.float64(83.33333333333334)]


In [126]:
df_[(df_['count'] != 5) & (df_['equal_yn'] != 1)]

,id,answer,result,count,equal_yn
8,72865629,1,2,3,0
21,75995516,1,0,3,0
38,79076968,1,2,4,0
29,78233699,1,0,4,0
32,78310990,1,2,3,0
2,70738995,2,1,4,0
20,77030147,2,1,3,0
24,78156592,0,1,4,0
34,79076968,1,2,4,0
10,73127128,0,1,3,0


In [1]:
import datetime

# 현재 시스템의 로컬 날짜와 시간 가져오기
now = datetime.datetime.now()
print(now)
# 출력 예시: 2026-07-17 14:48

2026-07-17 14:48:38.291279


In [4]:
if now.hour < 8:
    print("현재 시간은 오전 8시 이전입니다.")
    print(f"현재 시간: {now.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("현재 시간은 오전 8시 이후입니다.")
    print(f"현재 시간: {now.strftime('%Y-%m-%d %H:%M:%S')}")

현재 시간은 오전 8시 이후입니다.
현재 시간: 2026-07-17 14:48:38
